# Quantum Walks: Discreto y Continuo (DTQW y CTQW)

**Laboratorio 34 · Nivel muy avanzado**

Los *quantum walks* son la generalización cuántica de los paseos aleatorios clásicos.
Exhiben propagación **balística** ($\sigma \propto t$) frente a la difusión clásica
($\sigma \propto \sqrt{t}$), lo que da ventaja cuadrática en búsqueda y simulación.

Tipos estudiados:
- **DTQW** (Discrete-Time): moneda Hadamard + desplazamiento condicional
- **CTQW** (Continuous-Time): exponencial del laplaciano del grafo $e^{-iAt}$
- **Szegedy Walk**: walk en grafo bipartito, base para PageRank cuántico


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
import warnings; warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import QFT

print('Librerías cargadas OK')


## 1. Quantum Walk Discreto (DTQW) en línea 1D

El DTQW en línea usa un qubit de **moneda** y $n$ qubits de **posición**.

Un paso del walk:
1. **Moneda**: aplica $C = H$ (Hadamard) al qubit de moneda
2. **Desplazamiento**: condicional → mueve posición +1 (|0⟩) o -1 (|1⟩)

In [ ]:
class DTQW1D:
    """Quantum walk discreto en línea 1D de tamaño N (periódica)."""

    def __init__(self, N: int, estado_inicial='centro'):
        """
        N: número de posiciones (debe ser par)
        Estado en espacio (posición, moneda): dimensión 2N
        """
        self.N = N
        # Estado: vector de amplitudes en (posición × {0,1})
        # |ψ⟩ = Σ (α_{x,0}|x,0⟩ + α_{x,1}|x,1⟩)
        self.psi = np.zeros(2 * N, dtype=complex)
        x0 = N // 2  # posición central
        if estado_inicial == 'centro':
            self.psi[2*x0]   = 1.0 / np.sqrt(2)   # |x0, 0⟩
            self.psi[2*x0+1] = 1j / np.sqrt(2)    # |x0, 1⟩ (fase i → distribución simétrica)
        elif estado_inicial == 'localizado':
            self.psi[2*x0] = 1.0

    def paso(self):
        """Aplica un paso: moneda H + desplazamiento."""
        N = self.N
        psi_new = np.zeros(2 * N, dtype=complex)
        H = np.array([[1,1],[1,-1]]) / np.sqrt(2)  # moneda Hadamard

        for x in range(N):
            # Aplicar moneda
            coin_vec = self.psi[2*x:2*x+2].copy()
            coin_after = H @ coin_vec
            # Desplazamiento: |0⟩ → x+1, |1⟩ → x-1 (periódico)
            psi_new[2*((x+1) % N)]   += coin_after[0]
            psi_new[2*((x-1) % N)+1] += coin_after[1]

        self.psi = psi_new

    def probabilidades(self) -> np.ndarray:
        """Traza sobre el qubit de moneda."""
        N = self.N
        return np.array([abs(self.psi[2*x])**2 + abs(self.psi[2*x+1])**2 for x in range(N)])

# Simulación
N = 100
T_MAX = 50

walk = DTQW1D(N, 'centro')
snapshots = {}
for t in range(T_MAX + 1):
    if t in [0, 10, 25, 50]:
        snapshots[t] = walk.probabilidades().copy()
    if t < T_MAX:
        walk.paso()

print('Simulación DTQW completada')
posiciones = np.arange(N) - N//2
for t, probs in snapshots.items():
    sigma = np.sqrt(np.sum(probs * posiciones**2) - np.sum(probs * posiciones)**2)
    print(f'  t={t:3d}: σ={sigma:.2f} (balístico esperado: {t/np.sqrt(2):.2f})')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colores = ['#3498db','#e74c3c','#2ecc71','#f39c12']

for ax, (t, probs), color in zip(axes.flat, snapshots.items(), colores):
    ax.bar(posiciones, probs, width=0.8, color=color, alpha=0.8)
    sigma = np.sqrt(np.sum(probs * posiciones**2) - np.sum(probs * posiciones)**2)
    ax.set_title(f't = {t}  (σ = {sigma:.1f})')
    ax.set_xlabel('Posición'); ax.set_ylabel('Probabilidad')
    ax.set_xlim(-55, 55); ax.grid(alpha=0.3)

plt.suptitle('DTQW en línea 1D — Moneda Hadamard, estado inicial (|0⟩+i|1⟩)/√2', fontsize=11)
plt.tight_layout(); plt.show()


## 2. Velocidad de propagación: cuántico vs clásico

El walk cuántico se propaga balísticamente ($\sigma \propto t$, velocidad $v \approx 1/\sqrt{2}$)
mientras que el clásico difunde ($\sigma \propto \sqrt{t}$).

In [ ]:
# Comparativa cuántico vs clásico
T_range = np.arange(1, 61)
sigma_quantum = []
sigma_clasico = []

walk = DTQW1D(200, 'centro')
pos = np.arange(200) - 100
for t in T_range:
    walk.paso()
    probs = walk.probabilidades()
    sigma_quantum.append(np.sqrt(np.sum(probs * pos**2) - np.sum(probs * pos)**2))
    sigma_clasico.append(np.sqrt(t))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_range, sigma_quantum, 'b-', lw=2, label='DTQW (balístico: σ ∝ t)')
ax.plot(T_range, sigma_clasico, 'r--', lw=2, label='Clásico (difusión: σ ∝ √t)')
ax.plot(T_range, T_range/np.sqrt(2), 'b:', lw=1, alpha=0.5, label='Cota balística t/√2')
ax.set_xlabel('Pasos t'); ax.set_ylabel('Desviación estándar σ')
ax.set_title('Propagación cuántica vs clásica')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Ajuste lineal
from numpy.polynomial import polynomial as P
coef = np.polyfit(T_range, sigma_quantum, 1)
print(f'Velocidad balística medida: v = {coef[0]:.4f} (teórico: {1/np.sqrt(2):.4f})')


## 3. DTQW con diferentes monedas

La elección de la moneda afecta profundamente la distribución:
- **Hadamard**: asimétrica (sesgo a la izquierda)
- **(|0⟩+i|1⟩)/√2 inicial**: simétrica
- **Grover**: minimiza backtracking

In [ ]:
def dtqw_con_moneda(N: int, T: int, C: np.ndarray, x0_frac: float = 0.5) -> np.ndarray:
    """Simula DTQW con moneda C (2×2 unitaria)."""
    psi = np.zeros(2*N, dtype=complex)
    x0 = int(N * x0_frac)
    psi[2*x0] = 1/np.sqrt(2); psi[2*x0+1] = 1j/np.sqrt(2)

    for _ in range(T):
        psi_new = np.zeros(2*N, dtype=complex)
        for x in range(N):
            after = C @ psi[2*x:2*x+2]
            psi_new[2*((x+1)%N)]   += after[0]
            psi_new[2*((x-1)%N)+1] += after[1]
        psi = psi_new

    return np.array([abs(psi[2*x])**2 + abs(psi[2*x+1])**2 for x in range(N)])

N, T = 100, 40
pos = np.arange(N) - N//2

monedas = {
    'Hadamard': np.array([[1,1],[1,-1]])/np.sqrt(2),
    'Fourier':  np.array([[1,1j],[1j,1]])/np.sqrt(2),
    'Grover':   np.array([[-1,1],[1,-1]])/np.sqrt(2) + np.eye(2),
    'Y':        np.array([[0,-1],[1,0]], dtype=complex),
}
# Renormalizar Grover
monedas['Grover'] = np.array([[-1,1],[1,-1]])/np.sqrt(2)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (nombre, C) in zip(axes.flat, monedas.items()):
    probs = dtqw_con_moneda(N, T, C)
    sigma = np.sqrt(np.sum(probs * pos**2) - np.sum(probs * pos)**2)
    ax.bar(pos, probs, width=0.8, alpha=0.8)
    ax.set_title(f'Moneda {nombre} (σ={sigma:.1f})')
    ax.set_xlabel('Posición'); ax.set_ylabel('P(x)')
    ax.set_xlim(-55, 55); ax.grid(alpha=0.3)

plt.suptitle(f'DTQW con diferentes monedas — t={T} pasos', fontsize=11)
plt.tight_layout(); plt.show()


## 4. Quantum Walk Continuo (CTQW)

El CTQW en un grafo $G$ con matriz de adyacencia $A$ evoluciona como:

$$|\psi(t)\rangle = e^{-iAt}|\psi(0)\rangle$$

No necesita qubit de moneda — la superposición emerge de la estructura del grafo.

In [ ]:
class CTQW:
    """Quantum Walk Continuo en un grafo con matriz de adyacencia A."""

    def __init__(self, A: np.ndarray):
        self.A = A.astype(complex)
        self.N = A.shape[0]
        # Diagonalizar A para evolución eficiente
        self.eigvals, self.eigvecs = np.linalg.eigh(A)

    def evolucionar(self, psi0: np.ndarray, t: float) -> np.ndarray:
        """Evoluciona el estado psi0 hasta tiempo t."""
        # |ψ(t)⟩ = V · diag(e^{-iλt}) · V† · |ψ0⟩
        coeffs = self.eigvecs.conj().T @ psi0
        evolved = coeffs * np.exp(-1j * self.eigvals * t)
        return self.eigvecs @ evolved

    def probabilidades(self, psi0: np.ndarray, t: float) -> np.ndarray:
        return np.abs(self.evolucionar(psi0, t))**2


# Grafo: línea de N nodos
N = 50
A_linea = np.zeros((N, N))
for i in range(N-1):
    A_linea[i, i+1] = A_linea[i+1, i] = 1
# Periódico
A_linea[0, N-1] = A_linea[N-1, 0] = 1

walk_ct = CTQW(A_linea)
psi0 = np.zeros(N, dtype=complex)
psi0[N//2] = 1.0  # localizado en el centro

t_vals = [0, 5, 15, 30]
pos = np.arange(N) - N//2

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, t in zip(axes, t_vals):
    probs = walk_ct.probabilidades(psi0, t)
    sigma = np.sqrt(np.sum(probs * pos**2) - np.sum(probs * pos)**2)
    ax.bar(pos, probs, width=0.8, alpha=0.8, color='#8e44ad')
    ax.set_title(f't = {t}  (σ = {sigma:.1f})')
    ax.set_xlabel('Posición'); ax.set_ylabel('P(x)')
    ax.set_xlim(-28, 28); ax.grid(alpha=0.3)

plt.suptitle('CTQW en línea periódica de 50 nodos', fontsize=11)
plt.tight_layout(); plt.show()


## 5. CTQW en grafos: hipercubo y grafo aleatorio

La estructura del grafo determina si el walk puede llegar a todos los vértices
(ergodicidad) y con qué velocidad mezcla.

In [ ]:
# Hipercubo de dimensión 3 (8 nodos)
def hipercubo(n: int) -> np.ndarray:
    N = 2**n
    A = np.zeros((N, N), dtype=int)
    for i in range(N):
        for bit in range(n):
            j = i ^ (1 << bit)
            A[i, j] = 1
    return A

# Grafo de Petersen (10 nodos, 3-regular)
def petersen() -> np.ndarray:
    A = np.zeros((10,10), dtype=int)
    outer = [(0,1),(1,2),(2,3),(3,4),(4,0)]
    inner = [(5,7),(7,9),(9,6),(6,8),(8,5)]
    spokes = [(0,5),(1,6),(2,7),(3,8),(4,9)]
    for i,j in outer+inner+spokes:
        A[i,j]=A[j,i]=1
    return A

grafos = {
    'Línea 50n':    (A_linea, N//2),
    'Hipercubo 3D': (hipercubo(3), 0),
    'Petersen':     (petersen(), 0),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
T_mix = np.linspace(0, 20, 200)

for ax, (nombre, (A, nodo0)) in zip(axes, grafos.items()):
    Nn = A.shape[0]
    walk_g = CTQW(A)
    psi_g = np.zeros(Nn, dtype=complex); psi_g[nodo0] = 1.0

    # TVD vs distribución uniforme a lo largo del tiempo
    uniform = np.ones(Nn) / Nn
    tvds = []
    for t in T_mix:
        probs = walk_g.probabilidades(psi_g, t)
        tvds.append(0.5 * np.sum(np.abs(probs - uniform)))

    ax.plot(T_mix, tvds, lw=2)
    ax.set_xlabel('Tiempo t'); ax.set_ylabel('TVD vs uniforme')
    ax.set_title(f'Mezcla CTQW — {nombre}')
    ax.grid(alpha=0.3); ax.set_ylim(0, 1)

plt.suptitle('Mezcla del CTQW en distintos grafos', fontsize=11)
plt.tight_layout(); plt.show()


## 6. Implementación de CTQW en circuito cuántico

Para un grafo de línea de $2^n$ nodos, el CTQW puede implementarse eficientemente
mediante QFT + rotaciones de fase (los eigenestados del laplaciano son estados de Fourier).

In [ ]:
def ctqw_linea_circuito(n_qubits: int, t: float) -> QuantumCircuit:
    """
    Implementa e^{-iAt} para el grafo de línea periódica de 2^n nodos.
    Eigenvectores = estados de Fourier → QFT diagonaliza A.
    Eigenvalores: λ_k = 2cos(2πk/N)
    """
    N = 2**n_qubits
    qc = QuantumCircuit(n_qubits, name=f'CTQW(t={t:.2f})')

    # Paso 1: QFT†  (diagonaliza A)
    qc.append(QFT(n_qubits, do_swaps=True).inverse(), range(n_qubits))

    # Paso 2: fases e^{-iλ_k·t}
    for k in range(N):
        lam_k = 2 * np.cos(2 * np.pi * k / N)
        phase = -lam_k * t
        # Implementar la fase global para el estado |k⟩ (Gray code)
        # Aproximación: rotaciones Z individuales capturan la fase
        for q in range(n_qubits):
            bit = (k >> q) & 1
            if bit:
                qc.p(phase / n_qubits, q)

    # Paso 3: QFT
    qc.append(QFT(n_qubits, do_swaps=True), range(n_qubits))
    return qc

n = 3  # 8 nodos
t_circ = 5.0
qc_ctqw = ctqw_linea_circuito(n, t_circ)

print(f'Circuito CTQW (n={n}, t={t_circ}):')
print(f'  Qubits: {qc_ctqw.num_qubits}')
print(f'  Profundidad: {qc_ctqw.decompose().depth()}')
print(f'  Puertas: {qc_ctqw.decompose().size()}')
print()
print(qc_ctqw.draw('text'))


## 7. Aplicación: búsqueda cuántica con CTQW

Farhi & Gutmann (1998) mostraron que añadir un término oráculo al Hamiltoniano
permite buscar un vértice marcado en tiempo $O(\sqrt{N})$.

$$\hat{H}_{\text{búsqueda}} = -\gamma A - |w\rangle\langle w|$$

In [ ]:
def ctqw_busqueda(N: int, w: int, gamma: float, t: float) -> np.ndarray:
    """
    Búsqueda cuántica con CTQW en grafo completo K_N.
    H = -γ·A_KN - |w><w|  donde A_KN = J - I (J = todos-unos)

    Tiempo óptimo: t* ≈ π√N / 2,  γ = 1/N
    """
    J = np.ones((N, N)) - np.eye(N)  # adyacencia grafo completo
    oracle = np.zeros((N, N)); oracle[w, w] = 1.0
    H_search = -gamma * J - oracle

    # Estado inicial uniforme
    psi0 = np.ones(N, dtype=complex) / np.sqrt(N)

    # Evolución
    U = expm(-1j * H_search * t)
    psi_t = U @ psi0
    return np.abs(psi_t)**2

# Barrido en t para distintos N
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, N in zip(axes, [16, 64, 256]):
    w = 0
    gamma = 1.0 / N
    t_opt = np.pi * np.sqrt(N) / 2
    t_vals = np.linspace(0, 2 * t_opt, 150)

    p_target = [ctqw_busqueda(N, w, gamma, t)[w] for t in t_vals]

    ax.plot(t_vals, p_target, 'b-', lw=2)
    ax.axvline(t_opt, color='r', ls='--', alpha=0.7, label=f't*=π√N/2={t_opt:.1f}')
    ax.axhline(1, color='gray', ls=':', lw=0.8)
    ax.set_xlabel('Tiempo t'); ax.set_ylabel('P(vértice marcado)')
    ax.set_title(f'Búsqueda CTQW — N={N}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

    p_max = max(p_target)
    print(f'N={N:4d}: t*={t_opt:.2f}, P_max={p_max:.4f}')

plt.suptitle('Búsqueda cuántica con CTQW en grafo completo K_N', fontsize=11)
plt.tight_layout(); plt.show()


## 8. Comparativa DTQW vs CTQW vs Clásico


In [ ]:
comparativa = """
COMPARATIVA: PASEOS ALEATORIOS
═══════════════════════════════════════════════════════════════════

┌─────────────────┬───────────────┬────────────────┬─────────────────┐
│ Propiedad       │ Clásico       │ DTQW           │ CTQW            │
├─────────────────┼───────────────┼────────────────┼─────────────────┤
│ Propagación     │ σ ∝ √t        │ σ ∝ t          │ σ ∝ t           │
│ Moneda          │ Probabilística│ Unitaria (H)   │ No necesita     │
│ Tiempo discreto │ Sí            │ Sí             │ No (continuo)   │
│ Búsqueda        │ O(N)          │ O(√N) [2D]     │ O(√N) [K_N]     │
│ Mezcla en K_N   │ O(N log N)    │ O(1) [?]       │ O(√N)           │
│ Impl. circuito  │ No cuántico   │ n+1 qubits     │ QFT + fases     │
│ Aplicaciones    │ MCMC, difusión│ Elem. checking │ PageRank Q.     │
│                 │               │ Graph isomorf. │ Graph traversal │
└─────────────────┴───────────────┴────────────────┴─────────────────┘

VENTAJAS CUÁNTICAS:
  • Propagación balística → factor √N en búsqueda no estructurada (2D)
  • CTQW en K_N: búsqueda en O(√N) con γ=1/N (Childs & Goldstone 2004)
  • DTQW: base de algoritmos de element distinctness (O(N^{2/3}))
  • Walk cuántico → simulación de transporte cuántico en moléculas

LIMITACIONES:
  • Distribución no uniforme a tiempo largo (no mezcla ergódicamente)
  • DTQW requiere qubit de moneda extra (overhead de qubits)
  • CTQW en hardware: e^{-iAt} es difícil sin estructura del grafo
"""
print(comparativa)


## Conclusiones

- El **DTQW** propaga balísticamente gracias a la interferencia cuántica.
- La **moneda** controla la forma de la distribución; Hadamard da distribución asimétrica.
- El **CTQW** no necesita moneda y puede implementarse eficientemente vía QFT cuando el grafo tiene estructura de línea/hipercubo.
- La **búsqueda cuántica con CTQW** alcanza $P(w) \approx 1$ en tiempo $O(\sqrt{N})$.
- Los quantum walks son la base de algoritmos como element distinctness, graph traversal y transporte cuántico fotónico.